<a href="https://colab.research.google.com/github/HashiniDivyanjanee/Hashini-Divyanjanee-HPC-Task-2638122/blob/main/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%%writefile wordcount.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <pthread.h>

#define MAX_WORD_LEN 64
#define MAX_UNIQUE   10000

typedef struct {
    char word[MAX_WORD_LEN];
    int  count;
} WordCount;

typedef struct {
    int id;
    int start;   // first word index (inclusive)
    int end;     // last word index (exclusive)
} ThreadArg;

char **words = NULL;          // every word in the file
int totalWords = 0;

WordCount globalCounts[MAX_UNIQUE];   // shared result table
int globalUnique = 0;
pthread_mutex_t lock = PTHREAD_MUTEX_INITIALIZER;

// ---- worker thread ----
void *countWords(void *arg) {
    ThreadArg *a = (ThreadArg *)arg;
    WordCount *local = calloc(MAX_UNIQUE, sizeof(WordCount));
    int localUnique = 0;

    // 1. count inside this thread's own slice (no locking needed)
    for (int i = a->start; i < a->end; i++) {
        int found = 0;
        for (int j = 0; j < localUnique; j++) {
            if (strcmp(local[j].word, words[i]) == 0) {
                local[j].count++;
                found = 1;
                break;
            }
        }
        if (!found && localUnique < MAX_UNIQUE) {
            strncpy(local[localUnique].word, words[i], MAX_WORD_LEN - 1);
            local[localUnique].count = 1;
            localUnique++;
        }
    }

    // 2. merge into the shared table (protected by mutex)
    pthread_mutex_lock(&lock);
    for (int j = 0; j < localUnique; j++) {
        int found = 0;
        for (int k = 0; k < globalUnique; k++) {
            if (strcmp(globalCounts[k].word, local[j].word) == 0) {
                globalCounts[k].count += local[j].count;
                found = 1;
                break;
            }
        }
        if (!found && globalUnique < MAX_UNIQUE) {
            globalCounts[globalUnique] = local[j];
            globalUnique++;
        }
    }
    pthread_mutex_unlock(&lock);

    free(local);
    return NULL;
}

// sort by frequency (high - low), then alphabetically
int compare(const void *x, const void *y) {
    const WordCount *a = x, *b = y;
    if (b->count != a->count) return b->count - a->count;
    return strcmp(a->word, b->word);
}

int main(int argc, char *argv[]) {
    if (argc != 3) {
        printf("Usage: %s <number_of_threads> <input_file>\n", argv[0]);
        return 1;
    }
    int numThreads = atoi(argv[1]);
    if (numThreads <= 0) {
        printf("Number of threads must be a positive number\n");
        return 1;
    }

    FILE *fp = fopen(argv[2], "r");
    if (!fp) { perror("Cannot open input file"); return 1; }

    // read all words into memory
    int capacity = 1024;
    words = malloc(capacity * sizeof(char *));
    char buffer[MAX_WORD_LEN];
    while (fscanf(fp, "%63s", buffer) == 1) {
        if (totalWords == capacity) {
            capacity *= 2;
            words = realloc(words, capacity * sizeof(char *));
        }
        words[totalWords++] = strdup(buffer);
    }
    fclose(fp);
    if (totalWords == 0) { printf("Input file has no words\n"); return 0; }
    printf("Total words read: %d\n", totalWords);

    if (numThreads > totalWords) numThreads = totalWords;

    // dynamic slicing: slice sizes depend on word count and thread count
    pthread_t *threads = malloc(numThreads * sizeof(pthread_t));
    ThreadArg *args = malloc(numThreads * sizeof(ThreadArg));
    int base = totalWords / numThreads;
    int extra = totalWords % numThreads;
    int pos = 0;

    for (int i = 0; i < numThreads; i++) {
        int size = base + (i < extra ? 1 : 0);
        args[i].id = i;
        args[i].start = pos;
        args[i].end = pos + size;
        pos += size;
        printf("Thread %d -> words %d to %d\n", i, args[i].start, args[i].end - 1);
        if (pthread_create(&threads[i], NULL, countWords, &args[i]) != 0) {
            fprintf(stderr, "Error creating thread %d\n", i);
            return 1;
        }
    }
    for (int i = 0; i < numThreads; i++)
        pthread_join(threads[i], NULL);

    qsort(globalCounts, globalUnique, sizeof(WordCount), compare);

    FILE *out = fopen("result.txt", "w");
    if (!out) { perror("Cannot create result.txt"); return 1; }
    fprintf(out, "%-25s %s\n", "WORD", "FREQUENCY");
    fprintf(out, "--------------------------------------\n");
    long sum = 0;
    for (int i = 0; i < globalUnique; i++) {
        fprintf(out, "%-25s %d\n", globalCounts[i].word, globalCounts[i].count);
        sum += globalCounts[i].count;
    }
    fclose(out);
    printf("Unique words: %d, sum of counts: %ld\n", globalUnique, sum);
    printf("Results written to result.txt\n");

    pthread_mutex_destroy(&lock);
    for (int i = 0; i < totalWords; i++) free(words[i]);
    free(words); free(threads); free(args);
    return 0;
}

Writing wordcount.c


In [3]:
!gcc -pthread -O2 -o wordcount wordcount.c

In [6]:
!./wordcount 4 WordOccurrenceDataset.txt

Total words read: 120000
Thread 0 -> words 0 to 29999
Thread 1 -> words 30000 to 59999
Thread 2 -> words 60000 to 89999
Thread 3 -> words 90000 to 119999
Unique words: 94, sum of counts: 120000
Results written to result.txt


In [7]:
!head -20 result.txt

WORD                      FREQUENCY
--------------------------------------
student                   2456
teacher                   2413
linkedlist                1328
synchronisation           1323
assessment                1318
complexity                1317
network                   1317
Pthread                   1315
recursion                 1315
function                  1313
computer                  1309
security                  1309
runtime                   1308
array                     1307
learning                  1290
searching                 1286
sorting                   1285
syntax                    1285
